[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/05-explainability/03-phrase_fields_explained.ipynb)

In [1]:
# !pip install mbox

# PHRASE Fields Explained

`IndexType.PHRASE` is the type for anything with more than one word: names, addresses, descriptions. `01-index_types_and_recall_modes.ipynb` showed that `PHRASE` tolerates roughly as many character edits as `TERM` before a match disappears. What that comparison did not show is the more important difference: once a value has more than one word in it, `PHRASE` stops comparing character by character across the whole string, and starts comparing word by word. That single change is why `PHRASE` feels forgiving in ways `TERM` and `IDENT` simply are not.

In this notebook you will:

1. See that `APPROX` on a `PHRASE` field does not care about word order
2. See that a missing or an extra word costs far less than a single mistyped word would on a `TERM`-style comparison
3. Learn that this same word-based behavior carries over to `COMPLETE` and `DETECT`
4. See exactly what `EXACT` does and does not normalize before checking equality
5. Walk away with practical guidance for when `PHRASE`'s generosity helps you, and when it can quietly let through a false positive

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallMode

catalog = pd.read_csv("datasets/product_catalog.csv")
index = TableIndexer.create_index(
    catalog,
    index_columns=["product_id", "product_name", "category", "year_released", "price"],
    tmp_dir="tmp_index"
)

catalog[["product_id", "product_name"]]

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,product_id,product_name
0,B88-EXT-24,Extended Battery Pack Pro
1,A12-PWR-22,Portable Power Bank
2,C99-SNS-23,Motion Sensor Camera
3,D45-REL-21,Smart Relay Switch
4,E67-LEN-24,Wide Angle Camera Lens
5,F31-TRK-20,GPS Tracker Module
6,G14-CAB-19,Braided USB-C Cable
7,H82-DOC-23,Docking Station Hub
8,I29-BAT-22,Rechargeable Battery Cell
9,J56-SEN-24,Outdoor Motion Sensor


`product_name` is a `PHRASE` field, inferred automatically, four-word values like `"Extended Battery Pack Pro"`. Every query in this notebook searches `product_name` alone, using `modes={"product_name": ...}` to pin down exactly which comparison strategy is being tested.

## 1. Word order does not matter

Here is the same four words, in the original order and in three scrambled ones, all searched with `APPROX` against `"Extended Battery Pack Pro"`.

In [3]:
word_order_queries = [
    "Extended Battery Pack Pro",
    "Battery Extended Pack Pro",
    "Pack Pro Extended Battery",
    "Pro Pack Battery Extended",
]

rows = []
for query in word_order_queries:
    result = index.match(product_name=query, modes={"product_name": TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"query": query, "found": found, "product_name_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,query,found,product_name_score
0,Extended Battery Pack Pro,True,100
1,Battery Extended Pack Pro,True,100
2,Pack Pro Extended Battery,True,100
3,Pro Pack Battery Extended,True,100


Every scramble scores exactly `100`, identical to the original order. This would be a very strange result if `APPROX` were comparing the strings character by character, `"Pro Pack Battery Extended"` and `"Extended Battery Pack Pro"` share almost no character positions in common. It is not a strange result at all once you know `PHRASE` is comparing the *set* of words, not the sequence of characters. If your data is names people type in either order, or addresses where the house number sometimes comes first and sometimes last, this is precisely the behavior you want.

## 2. Missing and extra words cost less than a single typo

A `TERM` or `IDENT` field treats a dropped or inserted word as edits scattered across the whole string, expensive, and often fatal to the match. On a `PHRASE` field, dropping or adding a whole word is its own, separate kind of event, and it is considerably cheaper than you might expect.

In [4]:
structural_queries = [
    ("full match", "Extended Battery Pack Pro"),
    ("one word missing", "Extended Battery Pro"),
    ("one word added", "Extended Battery Pack Pro Max"),
    ("only one word", "Battery"),
    ("unrelated words, same length", "Xxxxxxxx Yyyyyyy Zzzz Ppp"),
]

rows = []
for label, query in structural_queries:
    result = index.match(product_name=query, modes={"product_name": TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"scenario": label, "query": query, "found": found, "product_name_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,scenario,query,found,product_name_score
0,full match,Extended Battery Pack Pro,True,100.0
1,one word missing,Extended Battery Pro,True,96.0
2,one word added,Extended Battery Pack Pro Max,True,69.0
3,only one word,Battery,True,89.0
4,"unrelated words, same length",Xxxxxxxx Yyyyyyy Zzzz Ppp,False,NaN


A dropped word barely registers. An added word costs more, but nowhere near as much as a `TERM` field would charge for scrambling that many characters. The one that should give you pause is `"only one word"`: searching for just `"Battery"`, a single word out of four, still returns `"Extended Battery Pack Pro"` with a strong score. `PHRASE`'s `APPROX` is, in effect, asking "how much of this query's word content is present in the indexed value," and one strong word out of one is a perfect answer to that narrower question, even though it is a weak answer to "is this the same product name."

The last row is the control: four words of the same lengths as the original, sharing no actual content, correctly finds nothing. `PHRASE` is still comparing real word content, not just word count or string length, it is simply comparing it more loosely than a character-by-character scan would.

## 3. `COMPLETE` and `DETECT` inherit the same word-based behavior

`04-recall-tuning/01-understanding_match_modes.ipynb` described `COMPLETE` as "does the query exist inside the indexed value." On a `PHRASE` field, "exist inside" turns out to mean "is this word, or these words, present," not "is this exact substring present at some exact character position." Watch what happens with a fragment that crosses a word boundary.

In [5]:
complete_queries = [
    ("two full words, in order", "battery pack"),
    ("two full words, reordered", "pack battery"),
    ("one full word", "pack"),
    ("a fragment crossing a word boundary", "attery pac"),
]

rows = []
for label, query in complete_queries:
    result = index.match(product_name=query, modes={"product_name": TableRecallMode.COMPLETE}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"scenario": label, "query": query, "found": found, "product_name_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,scenario,query,found,product_name_score
0,"two full words, in order",battery pack,True,91
1,"two full words, reordered",pack battery,True,91
2,one full word,pack,True,88
3,a fragment crossing a word boundary,attery pac,True,69


`"pack battery"`, reordered, scores identically to `"battery pack"` in order. That confirms `COMPLETE` on a `PHRASE` field is order-independent, the same way `APPROX` is. More surprising: `"attery pac"`, a fragment that starts and ends mid-word and matches no single complete word, still returns a match. `COMPLETE` on `PHRASE` is not the strict boolean substring check its name implies, it is a graded, word-aware containment score, closer in spirit to `APPROX` with a bias toward containment than to a literal `in` check on the raw string.

`DETECT`, the reverse direction, inherits the same tolerance. A longer query sentence is checked for the indexed phrase's *words*, not its exact substring, so it survives a missing word and a case change at the same time.

In [6]:
detect_queries = [
    "I need an Extended Battery Pack Pro for camping",
    "I need an extended battery pack for camping",   # lowercase, and "Pro" is missing
    "completely unrelated sentence about groceries",
]

rows = []
for query in detect_queries:
    result = index.match(product_name=query, modes={"product_name": TableRecallMode.DETECT}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"query": query, "found": found, "product_name_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,query,found,product_name_score
0,I need an Extended Battery Pack Pro for camping,True,88.0
1,I need an extended battery pack for camping,True,53.0
2,completely unrelated sentence about groceries,False,NaN


## 4. `EXACT`: strict equality, but "equal" is more forgiving than it sounds

`EXACT` still means what it says, a full match scores `100` and anything else scores `0`, no partial credit. What counts as "full," though, already includes some normalization: case and surrounding or doubled whitespace are folded away before the comparison happens, so `EXACT` is checking for the same words in the same order, not the exact same bytes.

In [7]:
exact_queries = [
    "Extended Battery Pack Pro",
    "extended battery pack pro",
    "EXTENDED BATTERY PACK PRO",
    "  Extended Battery Pack Pro ",
    "Extended  Battery Pack Pro",
    "Extended Battery Pack",   # missing "Pro"
]

rows = []
for query in exact_queries:
    result = index.match(product_name=query, modes={"product_name": TableRecallMode.EXACT}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"query": repr(query), "found": found, "product_name_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,query,found,product_name_score
0,'Extended Battery Pack Pro',True,100.0
1,'extended battery pack pro',True,100.0
2,'EXTENDED BATTERY PACK PRO',True,100.0
3,' Extended Battery Pack Pro ',True,100.0
4,'Extended Battery Pack Pro',True,100.0
5,'Extended Battery Pack',False,NaN


Case differences, leading and trailing spaces, and doubled internal spaces all still score `100`. Drop a whole word, and it drops straight to `0`, exactly as `EXACT` promises. If you need to also tolerate accents, umlauts, or other character-level normalization on top of this, that is what `CharacterMapping` in `02-data-harmonization/` is for, `EXACT`'s own normalization stops at case and whitespace.

## 5. Practical guidance for `PHRASE` fields

**Word order really can vary, and `PHRASE` already handles it.** Names typed last-first instead of first-last, addresses with the house number moved, multi-part titles reordered, none of these need special handling. This is the single biggest practical reason to choose `PHRASE` over `TERM` for anything with more than one word.

**A strong single-word overlap can produce a misleadingly high score.** Section 2 showed a one-word query scoring `89` against a four-word value. If your field can realistically be queried with just a fragment of the full value, do not rely on `APPROX` alone with a low `min_total_match_value`, pair it with a `minimum_quality` floor or a second field's weight, covered in `07-multi_field_weights_explained.ipynb`, so a single shared word cannot carry a whole match by itself.

**`COMPLETE` is a relevance signal, not a literal substring check.** If you need a true character-level substring test on a multi-word value, `PHRASE`'s `COMPLETE` will not give you one, it is word-aware and graded. `TERM` or `IDENT`, covered next, come much closer to literal containment, at the cost of losing `PHRASE`'s tolerance for reordering.

**Prefer `PHRASE` for names, descriptions, and addresses; prefer `TERM` or `IDENT` when the exact sequence of characters is the entire point.** The next two notebooks show exactly what you give up, and what you gain, by making that choice.

## Next steps

- **`04-term_fields_explained.ipynb`** - the same tests, on a field where word order and character sequence matter again
- **`05-ident_fields_explained.ipynb`** - identifiers, where even `APPROX`'s tolerance is deliberately tightened
- **`06-numeric_fields_explained.ipynb`** - what happens once the field is a number instead of a string

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*